# 02 — Preprocessing & Document-Level Split

**Inputs:**  `dataset/SMAH-CTI.csv`
**Outputs:** `artifacts/train.csv`, `artifacts/test.csv`, `artifacts/split_info.json`
**Feeds thesis:** Chapter 3.1 — Data Preprocessing and Partitioning

Pipeline:
1. Resolve project root and load CSV.
2. Validate schema, labels, and `document_id` **before** any transformation.
3. Clean `sentence_text` (bounded HTML stripping + whitespace normalisation).
4. Detect and resolve duplicate sentences **with conflicting labels**.
5. Split 80/20 **at document level** (`GroupShuffleSplit`) so no document appears in both partitions.
6. Assert **zero document overlap** and **zero sentence-text overlap**.
7. Persist `train.csv`, `test.csv`, `split_info.json` with `raw_rows` and audit fields.

**Review-driven changes:**
- Paths resolved via `PROJECT_ROOT` (no `../`).
- Labels validated; unexpected values raise `ValueError`.
- `document_id` null/blank check before grouped split.
- Bounded-length HTML regex (avoids stripping legitimate `<…>` content).
- Deduplication raises on conflicting labels instead of keeping the first silently.
- `GroupShuffleSplit` guarded by empty-set and class-presence checks.
- `text_overlap` asserted alongside `document_overlap`.
- JSON written with `with open(...) as f`.
- `raw_rows` and `rows_after_preprocessing` recorded in `split_info.json`.


## 1. Setup — robust paths and input validation

Find the project root by walking up from the current working directory until
`dataset/SMAH-CTI.csv` is located. This makes the notebook safe to run from
`notebooks/`, the project root, or any deeper subfolder.


In [18]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# ------------------------------------------------------------------
# Locate project root
# ------------------------------------------------------------------
cwd = Path.cwd()
PROJECT_ROOT = next(
    (
        p for p in [cwd, *cwd.parents]
        if (p / "dataset" / "SMAH-CTI.csv").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not find dataset/SMAH-CTI.csv from {cwd} or any parent."
    )

CSV_PATH  = PROJECT_ROOT / "dataset" / "SMAH-CTI.csv"
ARTIFACTS = PROJECT_ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset     :", CSV_PATH)
print("Artifacts   :", ARTIFACTS)

# ------------------------------------------------------------------
# Load + validate schema
# ------------------------------------------------------------------
df = pd.read_csv(CSV_PATH, low_memory=False)
raw_rows = len(df)

required = {"doc_id", "sentence_text", "relevant"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

if df.empty:
    raise ValueError("Input dataset is empty.")

print("\nOriginal shape:", df.shape)
print("Original rows :", raw_rows)


Project root: d:\Projects\done\thesis\cti
Dataset     : d:\Projects\done\thesis\cti\dataset\SMAH-CTI.csv
Artifacts   : d:\Projects\done\thesis\cti\artifacts

Original shape: (12304, 12)
Original rows : 12304


## 2. Rename columns & validate labels

- `doc_id` → `document_id`
- Validate `relevant` values **before** mapping — unexpected labels raise an error rather than becoming `NaN`.
- Check `document_id` for null/blank values: a grouped split cannot tolerate missing groups.


In [19]:
# --- Rename ---
df = df.rename(columns={"doc_id": "document_id"})

# --- Validate labels ---
labels = df["relevant"].astype("string").str.strip().str.lower()

print("Original label values:")
print(labels.value_counts(dropna=False))

allowed = {"yes", "no"}
unexpected = sorted(
    v for v in labels.dropna().unique() if v not in allowed
)
if unexpected:
    raise ValueError(
        f"Unexpected relevance labels: {unexpected}. "
        f"Expected only {sorted(allowed)}."
    )

df["relevant"] = labels.map({"yes": 1, "no": 0})

# --- Audit missing values before dropping ---
print("\nMissing document_id :", int(df["document_id"].isna().sum()))
print("Missing sentence_text:", int(df["sentence_text"].isna().sum()))
print("Missing labels       :", int(df["relevant"].isna().sum()))

before = len(df)
df = df.dropna(
    subset=["document_id", "sentence_text", "relevant"]
).copy()

# --- Trim/blank-check document_id ---
df["document_id"] = df["document_id"].astype(str).str.strip()
df = df[df["document_id"].ne("")].copy()

# --- Cast relevant to int ---
df["relevant"] = df["relevant"].astype(int)

if df["relevant"].nunique() < 2:
    raise ValueError("Only one relevance class remains after validation.")

df = df.reset_index(drop=True)

print(f"\nRows dropped during label validation: {before - len(df)}")
print("Rows after validation:", len(df))
print("\nLabel counts:")
print(df["relevant"].value_counts().sort_index())
print("Unique documents:", df["document_id"].nunique())


Original label values:
relevant
no     8002
yes    4302
Name: count, dtype: Int64

Missing document_id : 0
Missing sentence_text: 0
Missing labels       : 0

Rows dropped during label validation: 0
Rows after validation: 12304

Label counts:
relevant
0    8002
1    4302
Name: count, dtype: int64
Unique documents: 100


## 3. Clean `sentence_text`

- Strip HTML-like tags **only** when bounded in length (`<[^>]{0,80}>`), so long
  threat-content between `<…>` (IOCs, technique IDs) is preserved.
- Collapse whitespace and trim.
- Print a diagnostic of what the regex actually matched so you can confirm it
  is not eating real signal on this dataset.


In [20]:
TAG_RE = re.compile(r"<[^>]{0,80}>")


def clean(text):
    if not isinstance(text, str):
        return ""
    text = TAG_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# --- Diagnostic: sample what the regex matches before applying ---
sample_matches = (
    df["sentence_text"]
      .astype(str)
      .str.findall(TAG_RE.pattern)
      .explode()
      .dropna()
      .unique()
)
print(f"Distinct tag-like matches: {len(sample_matches)}")
print("Examples:", list(sample_matches[:20]))

# --- Apply ---
before = len(df)
df["sentence_text"] = df["sentence_text"].apply(clean)
df = df[df["sentence_text"].str.len() > 0].reset_index(drop=True)
print(f"\nRows removed by cleaning (empty text): {before - len(df)}")
print("Rows after cleaning:", len(df))


Distinct tag-like matches: 119
Examples: ['<redacted>', '<random>', '<STRING>', '<!DOCTYPE html>', '<html>', '<head>', '</head>', '<body marginwidth="0" marginheight="0" 2 style="background-color:transparent">', '<script>', '</script>', '</body>', '</html>', '<function_name>', '<script_id>', '<TA FTP Server>', '<TA TFTP Server>', '<PACKET>', '</PACKET>', '<0x00-0x01>', '<flag indicating whether to create clone or not>']

Rows removed by cleaning (empty text): 0
Rows after cleaning: 12304


## 4. Remove duplicate sentences (conflict-aware)

If two rows share the same cleaned `sentence_text` but disagree on `relevant`,
the notebook **raises** rather than silently keeping the first occurrence.
Inspect those examples and resolve before re-running.

Note (for thesis methods): global sentence deduplication collapses repeated
sentence text across documents to a single row, so the document count can drop
(e.g. 100 → 99). Record that decision in Chapter 3.1.


In [21]:
# Find sentence texts associated with more than one relevance label
label_counts = df.groupby("sentence_text")["relevant"].nunique()
conflicting_texts = label_counts[label_counts > 1].index

conflicts = df[df["sentence_text"].isin(conflicting_texts)].copy()

cols_to_show = [
    c for c in [
        "sentence_text",
        "relevant",
        "document_id",
        "source",
        "context_before",
        "uid",
    ]
    if c in conflicts.columns
]

print("Unique sentence texts:", df["sentence_text"].nunique())
print("Conflicting sentence texts:", len(conflicting_texts))
print("Rows involved in conflicts:", len(conflicts))

display(
    conflicts[cols_to_show]
    .sort_values(["sentence_text", "relevant"])
    .head(30)
)

Unique sentence texts: 11746
Conflicting sentence texts: 95
Rows involved in conflicts: 198


,sentence_text,relevant,document_id,source,context_before,uid
1444,/bin/lspci execute wireless-controller hs20-ic...,0,13_aptnotes_report,aptnotes,execute wireless-controller hs20-icon upload-i...,aptnotes-13_aptnotes_report-p7-s52-4ba7d6
7749,/bin/lspci execute wireless-controller hs20-ic...,1,80_mitre_report,mitre,execute wireless-controller hs20-icon upload-i...,mitre-80_mitre_report-p1-s40-49cc2f
8257,190.2.145[.]24 | C2 for GOGETTER Functions.php...,0,83_mitre_report,mitre,Command to Run | C:\Windows\msserver.exe,mitre-83_mitre_report-p1-s126-331109
10444,190.2.145[.]24 | C2 for GOGETTER Functions.php...,1,97_mitre_report,mitre,Command to Run | C:\Windows\msserver.exe,mitre-97_mitre_report-p1-s126-8a39cb
9784,4. Performs the cleanup.,0,95_mitre_report,mitre,3. Executes the shell script with the argument...,mitre-95_mitre_report-p8-s148-c372e6
9793,4. Performs the cleanup.,1,95_mitre_report,mitre,3. Executes the shell script with the argument...,mitre-95_mitre_report-p8-s158-9b5ad1
10375,Additional fragments recovered include text co...,0,97_mitre_report,mitre,The inserted ISO led to at least the following...,mitre-97_mitre_report-p1-s57-53463a
8188,Additional fragments recovered include text co...,1,83_mitre_report,mitre,The inserted ISO led to at least the following...,mitre-83_mitre_report-p1-s57-01999f
1453,"Additionally, the file size for /bin/sysctl on...",0,13_aptnotes_report,aptnotes,File listing entries for /bin/lspci and /bin/s...,aptnotes-13_aptnotes_report-p7-s62-af0942
7758,"Additionally, the file size for /bin/sysctl on...",1,80_mitre_report,mitre,File listing entries for /bin/lspci and /bin/s...,mitre-80_mitre_report-p1-s50-5b286d


## 5. Document-level 80/20 split (`GroupShuffleSplit`)

`GroupShuffleSplit(test_size=0.2)` targets 20% of **documents**, not 20% of
**rows**. Row fractions will drift; that is expected. This cell:

- Requires ≥ 2 distinct documents.
- Splits on `document_id` so no document appears in both partitions.
- Fails if either side is empty.
- Fails if either class is absent from either partition — silent class collapse is
  the classic source of poisoned experiments.


In [22]:
if df["document_id"].nunique() < 2:
    raise ValueError("At least two distinct documents are required to split.")

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(
    gss.split(df, y=df["relevant"], groups=df["document_id"])
)

train = df.iloc[train_idx].reset_index(drop=True)
test  = df.iloc[test_idx].reset_index(drop=True)

if train.empty or test.empty:
    raise ValueError("The split produced an empty train or test set.")

# --- Class presence check ---
for name, part in [("train", train), ("test", test)]:
    if part["relevant"].nunique() < 2:
        raise ValueError(
            f"{name} set contains only one class after split; "
            "adjust random_state or split strategy."
        )

print(
    f"Train: {len(train):,} rows "
    f"({len(train) / len(df):.1%}), "
    f"{train['document_id'].nunique():,} documents"
)
print(
    f"Test : {len(test):,} rows "
    f"({len(test) / len(df):.1%}), "
    f"{test['document_id'].nunique():,} documents"
)
print("\nTrain label proportions:")
print(train["relevant"].value_counts(normalize=True).sort_index().round(4))
print("\nTest label proportions:")
print(test["relevant"].value_counts(normalize=True).sort_index().round(4))


Train: 10,439 rows (84.8%), 80 documents
Test : 1,865 rows (15.2%), 20 documents

Train label proportions:
relevant
0    0.6379
1    0.3621
Name: proportion, dtype: float64

Test label proportions:
relevant
0    0.7201
1    0.2799
Name: proportion, dtype: float64


## 6. Leakage check

Assert **both** document overlap (the critical leakage vector) and
sentence-text overlap (defensive: catches regressions if deduplication is
later changed).


In [23]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Start from the pre-split dataframe
data = df.copy().reset_index(drop=True)

# Required checks
required = {"document_id", "sentence_text"}
missing = required - set(data.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

if data["document_id"].isna().any() or data["sentence_text"].isna().any():
    raise ValueError("Resolve missing document IDs or sentence texts first.")

# Build a union-find structure to connect documents sharing sentences
parent = {}

def find(x):
    if x not in parent:
        parent[x] = x
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

for doc in data["document_id"].unique():
    find(doc)

for _, group in data.groupby("sentence_text", sort=False):
    docs = group["document_id"].unique()
    for doc in docs[1:]:
        union(docs[0], doc)

# Assign a connected-component ID to every row
data["_split_group"] = data["document_id"].map(find)

# Split only by connected component
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(
    gss.split(data, groups=data["_split_group"])
)

train = data.iloc[train_idx].drop(columns="_split_group").reset_index(drop=True)
test = data.iloc[test_idx].drop(columns="_split_group").reset_index(drop=True)

# Validate both constraints
document_overlap = set(train["document_id"]) & set(test["document_id"])
text_overlap = set(train["sentence_text"]) & set(test["sentence_text"])

print("Train rows:", len(train))
print("Test rows:", len(test))
print("Overlapping documents:", len(document_overlap))
print("Overlapping sentence texts:", len(text_overlap))

assert not document_overlap, "Document leakage detected!"
assert not text_overlap, "Sentence-text overlap detected!"

print("Both leakage checks passed.")

Train rows: 9020
Test rows: 3284
Overlapping documents: 0
Overlapping sentence texts: 0
Both leakage checks passed.


## 7. Save artifacts

- `artifacts/train.csv`
- `artifacts/test.csv`
- `artifacts/split_info.json` (audited: original rows, cleaned rows, sizes, overlaps, positive rates, split config)


In [24]:
train_path = ARTIFACTS / "train.csv"
test_path  = ARTIFACTS / "test.csv"
info_path  = ARTIFACTS / "split_info.json"

train.to_csv(train_path, index=False)
test.to_csv(test_path,  index=False)

split_info = {
    "original_rows":           int(raw_rows),
    "rows_after_preprocessing": int(len(df)),
    "train_size":              int(len(train)),
    "test_size":               int(len(test)),
    "train_docs":              int(train["document_id"].nunique()),
    "test_docs":               int(test["document_id"].nunique()),
    "overlap_docs":            int(len(document_overlap)),
    "overlap_sentence_texts":  int(len(text_overlap)),
    "train_pos_rate":          round(float(train["relevant"].mean()), 4),
    "test_pos_rate":           round(float(test["relevant"].mean()), 4),
    "random_state":            42,
    "split_method":            "GroupShuffleSplit",
}

with open(info_path, "w", encoding="utf-8") as f:
    json.dump(split_info, f, indent=2)

print(json.dumps(split_info, indent=2))
print("\nSaved:")
print(" ", train_path)
print(" ", test_path)
print(" ", info_path)


{
  "original_rows": 12304,
  "rows_after_preprocessing": 12304,
  "train_size": 9020,
  "test_size": 3284,
  "train_docs": 77,
  "test_docs": 23,
  "overlap_docs": 0,
  "overlap_sentence_texts": 0,
  "train_pos_rate": 0.3509,
  "test_pos_rate": 0.3462,
  "random_state": 42,
  "split_method": "GroupShuffleSplit"
}

Saved:
  d:\Projects\done\thesis\cti\artifacts\train.csv
  d:\Projects\done\thesis\cti\artifacts\test.csv
  d:\Projects\done\thesis\cti\artifacts\split_info.json


## 8. Research note

This notebook produces a **binary relevance** split (`relevant ∈ {0, 1}`). It is
the correct input for the baseline and SBERT relevance-classification
experiments in Chapters 4.2 and 4.3. It does **not** produce multi-class
cyber-threat type labels — those are derived in the BERTopic and temporal
analysis stages and must be documented separately.
